# 🛡️ Notebook 2 — Fencing Tokens: making stale leaders harmless

## What you'll learn

- How a **fencing token** (also called *epoch* or *generation number*) works.
- A **BAD → BETTER → BEST** progression:
  1. **BAD:** no token at all (notebook 1).
  2. **BETTER:** token in memory, but storage forgets it on restart.
  3. **BEST:** token is persisted on the resource itself, and the resource rejects any write whose token is *less than or equal to* the highest seen.
- Why this is the pattern used by ZooKeeper (`zxid`), Kafka controller **epoch**, HBase region server epoch, Kubernetes `resourceVersion`, and HDFS NameNode.

## Analogy

Back to the bank. Every time a new manager is appointed, they get a **badge with a bigger number than the last one**. When a manager opens the vault, the vault reads the badge number and writes it down. It then refuses to open for any badge with a smaller or equal number.

Even if the old manager comes back with their old badge, the vault laughs and stays shut.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/split-brain-and-fencing
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🧱 A minimal lock service that hands out fencing tokens

Tokens must be **monotonic** — strictly increasing on every new grant. The lock service is the only place they're minted.


In [1]:
class LockService:
    """Hands out a strictly increasing token on each successful acquire."""

    def __init__(self):
        self._next = 0  # next token to give out
        self._holder = None

    def acquire(self, who: str) -> int:
        # In a real system this would be coordinated via Raft/Paxos/ZooKeeper.
        # Here we just pretend any acquire succeeds and bumps the token.
        self._next += 1
        self._holder = who
        print(f"  LockService: granted token {self._next} to {who}")
        return self._next


## 🟧 BETTER: token checked at the storage layer — but only in memory

Storage now remembers `highest_seen`. Writes with a smaller token are rejected.

**The subtle bug**: if storage *restarts* (crash, deploy, power cycle) it forgets `highest_seen` and accepts old tokens again. We'll hit that bug in a minute.


In [2]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class InMemoryStorage:
    highest_seen: int = 0  # ⚠️ not persisted
    log: List[str] = field(default_factory=list)

    def write(self, who: str, value: str, token: int) -> bool:
        if token < self.highest_seen:
            print(f"  ❌ REJECT {who}@{token}: stale (highest seen = {self.highest_seen})")
            return False
        self.highest_seen = token
        self.log.append(f"{who}@{token}: {value}")
        print(f"  ✅ accept  {who}@{token}: {value!r}")
        return True


lock = LockService()
store = InMemoryStorage()

tA = lock.acquire("A")
store.write("A", "config v1", tA)

tB = lock.acquire("B")          # A looked dead; B takes over with a bigger token
store.write("B", "config v2", tB)

# A wakes up from its GC pause, still thinks it is the leader, writes with OLD token
store.write("A", "config v1.1 (STALE)", tA)

print()
print("storage log:")
for e in store.log:
    print(" ", e)


  LockService: granted token 1 to A
  ✅ accept  A@1: 'config v1'
  LockService: granted token 2 to B
  ✅ accept  B@2: 'config v2'
  ❌ REJECT A@1: stale (highest seen = 2)

storage log:
  A@1: config v1
  B@2: config v2


Good — the stale write is blocked. 🎉

Now the **`BETTER` caveat**: imagine the storage node restarts right after `B` wrote `config v2`.


In [3]:
# Simulate a crash + restart: storage forgets highest_seen.
print("💥 storage crashes and restarts; it has no memory of the last token it saw")
store.highest_seen = 0

# A (still alive, still stale) retries its old write.
store.write("A", "config v1.1 (STALE, after restart)", tA)

print()
print("storage log after restart:")
for e in store.log:
    print(" ", e)


💥 storage crashes and restarts; it has no memory of the last token it saw
  ✅ accept  A@1: 'config v1.1 (STALE, after restart)'

storage log after restart:
  A@1: config v1
  B@2: config v2
  A@1: config v1.1 (STALE, after restart)


The stale write sneaked back in because `highest_seen` was only in RAM. This is exactly why real systems persist the token **on the same resource it protects** (disk, WAL, znode, etcd key).


## 🟩 BEST: persist the highest token next to the data

Two rules make this bulletproof:

1. Storage keeps `highest_seen` **on durable storage** (we simulate with a tiny JSON file).
2. Storage rejects tokens that are **≤** the highest seen. Using `<` would let the *same* token be replayed, which is a real attack / bug (e.g. retries from a stale leader).

We also make writes **atomic with the token update** — if the token is accepted, the write *and* the new `highest_seen` are persisted together.


In [4]:
import json, os, tempfile

class DurableStorage:
    """File-backed storage that fences writes by a monotonic token."""

    def __init__(self, path: str):
        self.path = path
        if not os.path.exists(path):
            self._save({"highest_seen": 0, "log": []})

    # --- tiny JSON persistence helpers (educational, not production) ---
    def _load(self) -> dict:
        with open(self.path) as f:
            return json.load(f)

    def _save(self, state: dict) -> None:
        # write-then-rename for atomicity
        d = os.path.dirname(self.path) or "."
        with tempfile.NamedTemporaryFile("w", dir=d, delete=False) as tmp:
            json.dump(state, tmp)
            tmp_path = tmp.name
        os.replace(tmp_path, self.path)

    # --- public API ---
    def highest_seen(self) -> int:
        return self._load()["highest_seen"]

    def write(self, who: str, value: str, token: int) -> bool:
        state = self._load()
        if token <= state["highest_seen"]:
            print(f"  ❌ REJECT {who}@{token}: need token > {state['highest_seen']}")
            return False
        state["highest_seen"] = token
        state["log"].append(f"{who}@{token}: {value}")
        self._save(state)
        print(f"  ✅ accept  {who}@{token}: {value!r}  (highest_seen -> {token})")
        return True

    def dump(self) -> None:
        for e in self._load()["log"]:
            print(" ", e)


In [5]:
import tempfile, pathlib

tmp = pathlib.Path(tempfile.mkdtemp()) / "store.json"
lock = LockService()
store = DurableStorage(str(tmp))

tA = lock.acquire("A")
store.write("A", "config v1", tA)

tB = lock.acquire("B")
store.write("B", "config v2", tB)

# Stale leader retries with old token — rejected.
store.write("A", "config v1.1 (STALE)", tA)

# Even replaying the latest token (<= highest_seen) is rejected — no replays.
store.write("B", "config v2 (duplicate)", tB)

print()
print("💥 simulate storage restart: new process, same file on disk")
store2 = DurableStorage(str(tmp))
print(f"  highest_seen survived restart: {store2.highest_seen()}")

# Stale leader tries again after restart — still rejected, because the token
# is persisted on the resource.
store2.write("A", "config v1.1 (STALE, after restart)", tA)

# A new legitimate leader C gets a bigger token and can write.
tC = lock.acquire("C")
store2.write("C", "config v3", tC)

print()
print("final storage log:")
store2.dump()


  LockService: granted token 1 to A
  ✅ accept  A@1: 'config v1'  (highest_seen -> 1)
  LockService: granted token 2 to B
  ✅ accept  B@2: 'config v2'  (highest_seen -> 2)
  ❌ REJECT A@1: need token > 2
  ❌ REJECT B@2: need token > 2

💥 simulate storage restart: new process, same file on disk
  highest_seen survived restart: 2
  ❌ REJECT A@1: need token > 2
  LockService: granted token 3 to C
  ✅ accept  C@3: 'config v3'  (highest_seen -> 3)

final storage log:
  A@1: config v1
  B@2: config v2
  C@3: config v3


### 🧠 Why the three invariants matter

| Invariant | Why |
|---|---|
| Tokens are **monotonically increasing** | Time never goes backwards — a bigger number means "more recent leadership". |
| The resource stores **its own** `highest_seen` | Leaders lie (by accident). The resource is the source of truth. |
| Check is **`≤`, not `<`** | Same-token replays from a stale leader are rejected. |
| `highest_seen` is **persisted** | A crash must not reopen the door to stale writers. |

Notice we never had to answer the impossible question *"is A really dead?"*. We only ever ask *"is this token fresh?"* — which has a cheap, local, deterministic answer.


## 🔬 Fix the bank example from Notebook 1


In [6]:
class FencedAccount:
    def __init__(self, owner: str, balance: int):
        self.owner = owner
        self.balance = balance
        self.highest_seen = 0

    def withdraw(self, who: str, amount: int, token: int) -> bool:
        if token <= self.highest_seen:
            print(f"  ❌ REJECT {who}@{token} withdraw {amount}: stale")
            return False
        self.highest_seen = token
        self.balance -= amount
        print(f"  ✅ {who}@{token} withdrew {amount:>4} | balance {self.balance}")
        return True


lock = LockService()
acct = FencedAccount("alice", 10_000)

# A acquires the lock and is about to process alice's $70 withdrawal,
# but pauses (GC / partition) BEFORE it actually runs the write.
tA = lock.acquire("A")

# Cluster assumes A is dead, promotes B. B processes the withdrawal.
tB = lock.acquire("B")
acct.withdraw("B", 7_000, tB)

# A wakes up, still thinks it's leader with token tA, runs the withdrawal.
# Without fencing this would double-charge alice. With fencing: rejected.
acct.withdraw("A", 7_000, tA)

print(f"\nfinal balance: {acct.balance} cents   (expected 3000)")


  LockService: granted token 1 to A
  LockService: granted token 2 to B
  ✅ B@2 withdrew 7000 | balance 3000
  ❌ REJECT A@1 withdraw 7000: stale

final balance: 3000 cents   (expected 3000)


Alice keeps her money. 🎉

> ℹ️ **Fencing vs idempotency — don't confuse them.** Fencing makes *stale leaders* harmless. It does **not** deduplicate retries from *legitimate* leaders (that's what idempotency keys / request IDs are for). The two patterns are complementary: fencing ensures "at most one leader writes at a time"; idempotency ensures "each logical request is applied at most once".

## 📚 Where this pattern shows up in real systems

- **Kafka** — the *controller epoch* is a fencing token. Brokers ignore commands from a controller with a smaller epoch.
- **ZooKeeper** — every transaction is stamped with a **`zxid`** (zookeeper transaction id), which increases monotonically.
- **HDFS** — NameNode generations; a standby that takes over gets a higher generation number written into the shared edits log.
- **HBase** — region servers have epoch numbers used during failover.
- **Kubernetes** — `resourceVersion` on objects prevents lost updates and stale writers (optimistic concurrency on top of etcd's `mod_revision`).
- **etcd** — every key-value has a `mod_revision`; callers use *compare-and-swap* on it, which is fencing under a different name.
- **Hazelcast / Redis Redlock (when combined with fencing)** — Martin Kleppmann's [famous critique](https://martin.kleppmann.com/2016/02/08/how-to-do-distributed-locking.html) of Redlock is literally "you also need a fencing token."

👉 Notebook 3 looks at the **other** tool in the toolkit: **STONITH & resource fencing** — what to do when the stale leader can misbehave in ways a token can't stop (e.g. it owns a shared disk, or it might issue side-effectful commands to external systems).
